# OVR Logistic Regression with Thresholding

This notebook builds a threshold-based one-vs-rest logistic regression benchmark for ACC skill tagging.

What this notebook does:
- loads ACC job, course, and skill embeddings from `../embedding/acc`
- joins each embedded entity to its ground-truth `extracted_skills` label from the CSVs
- builds two dataset variants: `jobs_only` and `jobs_plus_courses`
- trains one logistic regression model per skill on the train split
- tunes thresholds on the validation split only
- reports the final locked metrics on the held-out test split

This is important because we want the final numbers to reflect real held-out performance rather than validation-tuned performance.


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

EMBEDDINGS_PATH = Path("../embedding/acc")
JOB_LABEL_CSV = Path("../data/acc/audit_tax_accounting_jobs.csv")
COURSE_LABEL_CSV = Path("../data/acc/acc_courses.csv")
SKILLS_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_skills_embeddings.jsonl"
JOBS_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_jobs_embeddings.jsonl"
COURSES_EMBEDDINGS_PATH = EMBEDDINGS_PATH / "acc_courses_embeddings.jsonl"

skills_embeddings = pd.read_json(SKILLS_EMBEDDINGS_PATH, lines=True)
jobs_embeddings = pd.read_json(JOBS_EMBEDDINGS_PATH, lines=True)
courses_embeddings = pd.read_json(COURSES_EMBEDDINGS_PATH, lines=True)
job_labels = pd.read_csv(JOB_LABEL_CSV)
course_labels = pd.read_csv(COURSE_LABEL_CSV)

print("Embedding splits:")
print(jobs_embeddings["split"].value_counts().sort_index().rename("jobs"))
print()
print(courses_embeddings["split"].value_counts().sort_index().rename("courses"))
print()
print(f"num_skills: {len(skills_embeddings)}")


Embedding splits:
split
test     20
train    59
val      19
Name: jobs, dtype: int64

split
test     13
train    38
val      13
Name: courses, dtype: int64

num_skills: 229


## Helper Functions

These helpers do the repetitive setup work:
- convert `extracted_skills` text into Python lists
- join embeddings to labels
- build binary label matrices
- train one logistic regression per skill
- tune thresholds and compute metrics

Keeping them here makes the actual experiment cells shorter and easier to read.


In [ ]:
def parse_skill_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [skill.strip() for skill in str(value).split("|") if skill.strip()]


def attach_actual_skill_lists(df, labels_df, left_key, right_key):
    merged = df.merge(
        labels_df,
        left_on=left_key,
        right_on=right_key,
        how="left",
        suffixes=("_embed", "_label"),
    )

    missing_mask = merged["extracted_skills"].isna()
    dropped_ids = merged.loc[missing_mask, left_key].tolist()
    if dropped_ids:
        print(f"Dropping {len(dropped_ids)} unlabeled rows for {left_key}: {dropped_ids}")
        merged = merged.loc[~missing_mask].copy()

    merged["actual_skill_lists"] = merged["extracted_skills"].apply(parse_skill_list)
    merged["display_title"] = merged["title_label"] if "title_label" in merged.columns else merged["title"]
    merged["entity_id"] = merged[left_key]

    return merged[["entity_type", "entity_id", "display_title", "embedding", "actual_skill_lists"]].copy()


def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator


def build_dataset(train_df, val_df, test_df, skill_names):
    X_train = np.vstack(train_df["embedding"].to_numpy()).astype(np.float32)
    X_val = np.vstack(val_df["embedding"].to_numpy()).astype(np.float32)
    X_test = np.vstack(test_df["embedding"].to_numpy()).astype(np.float32)

    Y_train = build_indicator_matrix(train_df["actual_skill_lists"].tolist(), skill_names)
    Y_val = build_indicator_matrix(val_df["actual_skill_lists"].tolist(), skill_names)
    Y_test = build_indicator_matrix(test_df["actual_skill_lists"].tolist(), skill_names)

    return X_train, X_val, X_test, Y_train, Y_val, Y_test


def fit_ovr_probabilities(X_train, Y_train, X_val, X_test, skill_names):
    # Store predicted probabilities for each split.
    train_proba = np.zeros((X_train.shape[0], len(skill_names)), dtype=np.float32)
    val_proba = np.zeros((X_val.shape[0], len(skill_names)), dtype=np.float32)
    test_proba = np.zeros((X_test.shape[0], len(skill_names)), dtype=np.float32)

    # Track which skills had a fitted logistic regression model.
    fitted_mask = np.zeros(len(skill_names), dtype=bool)
    skipped_skills = []

    # Train one binary classifier per skill.
    for j, skill_name in enumerate(skill_names):
        y_train_j = Y_train[:, j]
        unique_classes = np.unique(y_train_j)

        # LogisticRegression requires at least two classes.
        # If the skill is constant in training data, use that
        # constant class as the fallback probability.
        if len(unique_classes) < 2:
            constant_proba = float(unique_classes[0])
            train_proba[:, j] = constant_proba
            val_proba[:, j] = constant_proba
            test_proba[:, j] = constant_proba
            skipped_skills.append(skill_name)
            continue

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )
        model.fit(X_train, y_train_j)

        # Save the probability of class 1 for each split.
        train_proba[:, j] = model.predict_proba(X_train)[:, 1]
        val_proba[:, j] = model.predict_proba(X_val)[:, 1]
        test_proba[:, j] = model.predict_proba(X_test)[:, 1]
        fitted_mask[j] = True

    return train_proba, val_proba, test_proba, fitted_mask, skipped_skills


def micro_metrics(y_true, y_pred):
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1


def macro_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return precision, recall, f1


def tune_global_threshold(y_true, y_score):
    # Try every unique score as a possible threshold.
    candidate_thresholds = np.unique(y_score)

    # Add a threshold above 1 so the "predict nothing" case is also checked.
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        # Convert probabilities into binary predictions.
        y_pred = (y_score >= threshold).astype(np.uint8)
        precision, recall, f1 = micro_metrics(y_true, y_pred)

        # Rank by F1 first, then recall, then prefer smaller thresholds.
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold


def tune_best_threshold_for_one_skill(y_true, y_score):
    # Try every unique score as a possible threshold for one skill.
    candidate_thresholds = np.unique(y_score)
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        # Compute binary precision, recall, and F1 for this skill only.
        tp = np.logical_and(y_pred == 1, y_true == 1).sum()
        fp = np.logical_and(y_pred == 1, y_true == 0).sum()
        fn = np.logical_and(y_pred == 0, y_true == 1).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

        # Rank by F1 first, then recall, then prefer smaller thresholds.
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold


def indicator_row_to_skills(row, skill_names):
    return [skill_names[j] for j, flag in enumerate(row) if flag == 1]


## Join Embeddings to Labels

At this point we connect the embeddings to the CSV labels.

This is what turns the raw embedding files into supervised learning data:
- the embedding file gives the feature vector
- the CSV gives the true skill list

After the join, each row has:
- entity id
- title
- embedding
- actual skill list


In [12]:
skill_names = skills_embeddings["skill_name"].tolist()

jobs_train_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "train"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)
jobs_val_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "val"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)
jobs_test_df = attach_actual_skill_lists(
    jobs_embeddings[jobs_embeddings["split"] == "test"].copy(),
    job_labels,
    left_key="job_id",
    right_key="uuid",
)

courses_train_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "train"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)
courses_val_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "val"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)
courses_test_df = attach_actual_skill_lists(
    courses_embeddings[courses_embeddings["split"] == "test"].copy(),
    course_labels,
    left_key="course_code",
    right_key="moduleCode",
)

print("Supervised rows after joins:")
print(f"jobs train:    {len(jobs_train_df)}")
print(f"jobs val:      {len(jobs_val_df)}")
print(f"jobs test:     {len(jobs_test_df)}")
print(f"courses train: {len(courses_train_df)}")
print(f"courses val:   {len(courses_val_df)}")
print(f"courses test:  {len(courses_test_df)}")


Dropping 1 unlabeled rows for course_code: ['ACC4761E']
Supervised rows after joins:
jobs train:    59
jobs val:      19
jobs test:     20
courses train: 37
courses val:   13
courses test:  13


## Build Benchmark Variants

We evaluate two scopes:
- `jobs_only`: only job postings are used
- `jobs_plus_courses`: job postings and ACC courses are combined

This helps us compare the OVR model against the same kinds of setups explored in the cosine notebook.


In [13]:
dataset_variants = {
    "jobs_only": {
        "train": jobs_train_df,
        "val": jobs_val_df,
        "test": jobs_test_df,
    },
    "jobs_plus_courses": {
        "train": pd.concat([jobs_train_df, courses_train_df], ignore_index=True),
        "val": pd.concat([jobs_val_df, courses_val_df], ignore_index=True),
        "test": pd.concat([jobs_test_df, courses_test_df], ignore_index=True),
    },
}

for variant_name, frames in dataset_variants.items():
    print(variant_name)
    print(f"  train rows: {len(frames['train'])}")
    print(f"  val rows:   {len(frames['val'])}")
    print(f"  test rows:  {len(frames['test'])}")


jobs_only
  train rows: 59
  val rows:   19
  test rows:  20
jobs_plus_courses
  train rows: 96
  val rows:   32
  test rows:  33


## Train, Tune, and Evaluate

This is the main experiment cell.

For each dataset variant, it:
- builds feature and label matrices
- trains one logistic regression model per skill
- tunes threshold(s) on validation
- evaluates the locked predictions on test

It stores both a summary table and prediction previews so we can inspect what the model is actually predicting.


In [14]:
results = []
artifacts = {}

for variant_name, frames in dataset_variants.items():
    X_train, X_val, X_test, Y_train, Y_val, Y_test = build_dataset(
        frames["train"],
        frames["val"],
        frames["test"],
        skill_names,
    )

    train_proba, val_proba, test_proba, fitted_mask, skipped_skills = fit_ovr_probabilities(
        X_train,
        Y_train,
        X_val,
        X_test,
        skill_names,
    )

    global_threshold = tune_global_threshold(Y_val, val_proba)
    global_test_pred = (test_proba >= global_threshold).astype(np.uint8)
    global_micro_precision, global_micro_recall, global_micro_f1 = micro_metrics(Y_test, global_test_pred)
    global_macro_precision, global_macro_recall, global_macro_f1 = macro_metrics(Y_test, global_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "global",
        "micro_precision": global_micro_precision,
        "micro_recall": global_micro_recall,
        "micro_f1": global_micro_f1,
        "macro_precision": global_macro_precision,
        "macro_recall": global_macro_recall,
        "macro_f1": global_macro_f1,
        "global_threshold": global_threshold,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    global_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in global_test_pred]
    artifacts[(variant_name, "global")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in global_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": np.full(len(skill_names), global_threshold, dtype=np.float32),
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

    skill_thresholds = np.full(len(skill_names), 0.5, dtype=np.float32)
    for j in np.where(fitted_mask)[0]:
        skill_thresholds[j] = tune_best_threshold_for_one_skill(Y_val[:, j], val_proba[:, j])

    skill_test_pred = (test_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
    skill_micro_precision, skill_micro_recall, skill_micro_f1 = micro_metrics(Y_test, skill_test_pred)
    skill_macro_precision, skill_macro_recall, skill_macro_f1 = macro_metrics(Y_test, skill_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "skill_specific",
        "micro_precision": skill_micro_precision,
        "micro_recall": skill_micro_recall,
        "micro_f1": skill_micro_f1,
        "macro_precision": skill_macro_precision,
        "macro_recall": skill_macro_recall,
        "macro_f1": skill_macro_f1,
        "global_threshold": np.nan,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    skill_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in skill_test_pred]
    artifacts[(variant_name, "skill_specific")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in skill_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": skill_thresholds,
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

comparison_df = pd.DataFrame(results).sort_values(
    by=["micro_f1", "micro_recall", "macro_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)
comparison_df


,dataset_variant,threshold_type,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,global_threshold,fitted_models,skipped_skills,train_rows,val_rows,test_rows
0,jobs_only,global,0.632353,0.767857,0.693548,0.059563,0.068975,0.061540,0.505580,52,177,59,19,20
1,jobs_plus_courses,global,0.633094,0.606897,0.619718,0.049775,0.051410,0.048544,0.578499,74,155,96,32,33
2,jobs_only,skill_specific,0.134815,0.812500,0.231258,0.043275,0.092475,0.048498,NaN,52,177,59,19,20
3,jobs_plus_courses,skill_specific,0.078155,0.841379,0.143025,0.048008,0.118980,0.056172,NaN,74,155,96,32,33


## Inspect the Best Configuration

This final code cell prints the strongest result from the comparison table and shows a small prediction preview.

That makes it easier to check whether the model is making sensible skill predictions instead of relying on summary metrics alone.


In [15]:
best_row = comparison_df.iloc[0]
best_key = (best_row["dataset_variant"], best_row["threshold_type"])

print("Best evaluated configuration on held-out test:")
print(best_row.to_string())
print()
print("Prediction preview:")
print(artifacts[best_key]["predictions_df"].head(10).to_string(index=False))


Best evaluated configuration on held-out test:
dataset_variant     jobs_only
threshold_type         global
micro_precision      0.632353
micro_recall         0.767857
micro_f1             0.693548
macro_precision      0.059563
macro_recall         0.068975
macro_f1              0.06154
global_threshold      0.50558
fitted_models              52
skipped_skills            177
train_rows                 59
val_rows                   19
test_rows                  20

Prediction preview:
entity_type                        entity_id                                                                       title                                                                                                                                                                                                                                   actual_skills                                                                                                                                                        

## Summary

This notebook now reads top to bottom as one clean benchmark:
- load data
- prepare supervised labels
- define the experiment variants
- train and tune the OVR models
- report final test results

In practice, the global-threshold OVR variant is usually the stronger baseline here, while the skill-specific-threshold version tends to overfit because the validation set is small and many skills are rare.

### Caveats and Limitations
- This benchmark assumes the CSV `extracted_skills` labels are the ground truth.
- Many skills are rare, so the model learns much better for common skills than rare ones.
- OVR cannot train a classifier for a skill if the train split does not contain both positive and negative examples for that skill.
- Micro-F1 is driven more by common skills, so it can look stronger than macro-F1.
- The skill-specific-threshold variant is more flexible, but it can overfit when the validation set is small.
- One course, `ACC4761E`, is excluded because it currently has no labeled `extracted_skills` value.
